# 🤖 GPT Training on Google Colab (GPU)
**Seedha upload karo aur Run All dabao — bas itna hi!**

Steps:
1. ✅ GPU check
2. ✅ GitHub se code download
3. ✅ PyTorch files replace
4. ✅ Training shuru
5. ✅ Brain save + download

In [ ]:
# ════════════════════════════════════════════════
# CELL 1 — GPU Check
# ════════════════════════════════════════════════
import torch

if torch.cuda.is_available():
    print(f'✅ GPU mil gaya: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ GPU nahi mila!')
    print('   Runtime → Change runtime type → T4 GPU → Save karo, phir dobara run karo')

In [ ]:
# ════════════════════════════════════════════════
# CELL 2 — GitHub se code clone karo
# ════════════════════════════════════════════════
import os

# 👇 APNA GITHUB REPO URL YAHAN DAALO
GITHUB_REPO = 'https://github.com/niteenmaurya/Colz.git'
FOLDER      = '/content/gpt'

if os.path.exists(FOLDER):
    print('Folder pehle se hai, update kar raha hoon...')
    os.system(f'cd {FOLDER} && git pull')
else:
    os.system(f'git clone {GITHUB_REPO} {FOLDER}')

os.chdir(FOLDER)
print(f'\n📁 Files in {FOLDER}:')
print('  ' + '\n  '.join(sorted(os.listdir())))

In [ ]:
# ════════════════════════════════════════════════
# CELL 3 — transformer.py likhna (PyTorch version)
# ════════════════════════════════════════════════
transformer_code = '''
import torch
import torch.nn as nn
import math

DEVICE = (
    torch.device("cuda")  if torch.cuda.is_available() else
    torch.device("mps")   if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"[Transformer] Device: {DEVICE}")

class _CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len, d_model = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(d_model, dim=-1)
        q = q.view(seq_len, self.n_heads, self.d_head).transpose(0, 1)
        k = k.view(seq_len, self.n_heads, self.d_head).transpose(0, 1)
        v = v.view(seq_len, self.n_heads, self.d_head).transpose(0, 1)
        scale  = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale
        mask   = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=x.device), diagonal=1)
        scores = scores + mask
        attn   = torch.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)
        out    = out.transpose(0, 1).contiguous().view(seq_len, d_model)
        return self.out_proj(out)

class _FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class _TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = _CausalSelfAttention(d_model, n_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = _FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len
        self.token_emb   = nn.Embedding(vocab_size, d_model)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        self.blocks      = nn.ModuleList([_TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_final    = nn.LayerNorm(d_model)
        self.lm_head     = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
        self.to(DEVICE)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, token_ids):
        if isinstance(token_ids, list):
            ids = torch.tensor(token_ids, dtype=torch.long, device=DEVICE)
        else:
            ids = token_ids.to(DEVICE)
        seq_len = ids.shape[0]
        if seq_len > self.max_seq_len:
            ids = ids[-self.max_seq_len:]
            seq_len = self.max_seq_len
        pos    = torch.arange(seq_len, device=DEVICE)
        x      = self.token_emb(ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x      = self.ln_final(x)
        logits = self.lm_head(x)
        return logits.tolist()

    def forward_tensor(self, ids):
        seq_len = ids.shape[0]
        if seq_len > self.max_seq_len:
            ids = ids[-self.max_seq_len:]
            seq_len = self.max_seq_len
        pos = torch.arange(seq_len, device=ids.device)
        x   = self.token_emb(ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_final(x))

    def backward(self, dlogits): pass
    def zero_grad(self):
        for p in self.parameters():
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()
    def all_params_and_grads(self):
        return [(p.tolist(), (p.grad.tolist() if p.grad is not None else []), n)
                for n, p in self.named_parameters()]
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())
'''

with open('transformer.py', 'w') as f:
    f.write(transformer_code)
print('✅ transformer.py likha gaya (PyTorch GPU version)')

In [ ]:
# ════════════════════════════════════════════════
# CELL 4 — trainer.py (तेज़ GPU Batching वर्ज़न) ⚡
# ════════════════════════════════════════════════
trainer_code = '''
import math, random, torch, torch.nn as nn
import torch.optim as optim

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# (पुराने वर्ज़न के लिए खाली फ़ंक्शन्स ताकि एरर न आए)
def cross_entropy_loss(logits, targets): return 0.0, []
class AdamOptimizer:
    def __init__(self, lr=3e-4, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01): pass
    def step(self, params_and_grads): pass
def clip_gradients(params_and_grads, max_norm=1.0): return max_norm

class Trainer:
    def __init__(self, model, lr=3e-4, max_norm=1.0):
        self.model    = model
        self.max_norm = max_norm
        self.torch_optimizer = optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.999), weight_decay=0.01)
        self.loss_history = []
        self._loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    def train_step(self, batch_windows):
        self.torch_optimizer.zero_grad()
        total_loss = 0.0
        
        # 32 का बैच एक-एक करके लेंगे, पर कैलकुलेशन एक साथ (Batching/Accumulation)
        for token_ids in batch_windows:
            if len(token_ids) < 2: continue
            inputs  = token_ids[:-1]
            targets = token_ids[1:]
            inp_t = torch.tensor(inputs,  dtype=torch.long,  device=DEVICE)
            tgt_t = torch.tensor(targets, dtype=torch.long,  device=DEVICE)
            
            with torch.enable_grad():
                logits = self.model.forward_tensor(inp_t)
                loss   = self._loss_fn(logits, tgt_t[:logits.shape[0]])
                
                # बैच के हिसाब से लॉस को बाँट कर backward करना
                loss = loss / len(batch_windows)
                loss.backward()
                
            total_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_norm)
        self.torch_optimizer.step()
        return total_loss

    def train(self, all_token_ids, seq_len, epochs, log_every=10):
        windows = [all_token_ids[s:s+seq_len+1] for s in range(0, len(all_token_ids)-seq_len, seq_len)]
        if not windows: return
        
        batch_size = 32  # 🚀 GPU को स्पीड देने के लिए बैच साइज़
        scheduler = optim.lr_scheduler.CosineAnnealingLR(self.torch_optimizer, T_max=epochs, eta_min=1e-5)
        
        print(f"[Trainer] Fast Batching Mode ON! device={DEVICE}")
        self.model.train()
        
        for epoch in range(1, epochs+1):
            random.shuffle(windows)
            batches = [windows[i:i + batch_size] for i in range(0, len(windows), batch_size)]
            
            epoch_loss = sum(self.train_step(b) for b in batches)
            avg_loss   = epoch_loss / len(batches)
            self.loss_history.append(avg_loss)
            scheduler.step()
            
            if epoch % log_every == 0 or epoch == 1:
                ppl = math.exp(min(avg_loss, 20))
                print(f"  epoch {epoch:4d}/{epochs}  |  loss={avg_loss:.4f}  |  ppl={ppl:.2f}")
                
        self.model.eval()
'''

with open('trainer.py', 'w') as f:
    f.write(trainer_code)
print('✅ trainer.py FAST version (Batching) बन गया!')

In [ ]:
# ════════════════════════════════════════════════
# CELL 5 — data.txt check (nahi hai toh sample banao)
# ════════════════════════════════════════════════
import os

if not os.path.exists('data.txt'):
    print('⚠️  data.txt nahi mila — sample text bana raha hoon...')
    sample = (
        'The cat sat on the mat. The dog ran in the park.\n'
        'A cat and a dog became good friends one day.\n'
        'The cat chased the dog across the green field.\n'
        'Once upon a time there lived a clever cat.\n'
        'The dog was very fast and the cat was very smart.\n'
        'Together they explored the forest every morning.\n'
        'The sun rose over the hills and the birds sang.\n'
        'A big brown dog sat next to a small orange cat.\n'
    ) * 200  # 200 baar repeat = achha corpus
    with open('data.txt', 'w') as f:
        f.write(sample)
    print(f'✅ Sample data.txt banaya: {len(sample):,} characters')
else:
    with open('data.txt') as f:
        content = f.read()
    print(f'✅ data.txt mila: {len(content):,} characters')

In [ ]:
# ════════════════════════════════════════════════
# CELL 6 — TRAINING SHURU!  ⚡
# ════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/gpt')

# Reload modules (agar pehle se load hain)
import importlib
for mod in ['tokenizer', 'transformer', 'trainer', 'inference']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from tokenizer  import CharTokenizer
from transformer import GPT
from trainer    import Trainer
from inference  import generate

# ── Corpus ────────────────────────────────────────────────────────────────────
with open('data.txt', 'r', encoding='utf-8') as f:
    CORPUS = f.read()

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tok = CharTokenizer()
tok.build_vocab(CORPUS)
tok.save('vocab.json')
print(f'Vocab size: {tok.vocab_size}')
all_ids = tok.encode(CORPUS)

# ── Model ─────────────────────────────────────────────────────────────────────
# 👇 Chota corpus? d_model=64, n_layers=2 rakhna
# 👇 Bada corpus?  d_model=128, n_layers=4 kar sakte ho
model = GPT(
    vocab_size  = tok.vocab_size,
    d_model     = 128,
    n_heads     = 4,
    n_layers    = 4,
    d_ff        = 512,
    max_seq_len = 128,
)
print(f'Model parameters: {model.count_parameters():,}')

# ── Train ─────────────────────────────────────────────────────────────────────
trainer = Trainer(model, lr=3e-4)
trainer.train(all_ids, seq_len=128, epochs=3000, log_every=100)

In [ ]:
# ════════════════════════════════════════════════
# CELL 7 — Text Generate karo
# ════════════════════════════════════════════════
model.eval()

prompts = ['the cat', 'the dog', 'once upon', 'a cat']
print('=' * 55)
print('GENERATED TEXT')
print('=' * 55)
for p in prompts:
    out = generate(model, tok, p, max_new=80, strategy='top_k', k=5, temperature=0.8)
    print(f"\n📝 Prompt : '{p}'")
    print(f"   Output : {out}")

In [ ]:
# ════════════════════════════════════════════════
# CELL 8 — Brain save + Download
# ════════════════════════════════════════════════
import torch, pickle

# PyTorch proper save
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size' : tok.vocab_size,
        'd_model'    : 128,
        'n_heads'    : 4,
        'n_layers'   : 4,
        'd_ff'       : 512,
        'max_seq_len': 128,
    },
    'loss_history': trainer.loss_history,
}, 'brain.pt')

# Pickle backup
with open('brain.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ brain.pt   — saved (PyTorch format, recommended)')
print('✅ brain.pkl  — saved (pickle backup)')
print('✅ vocab.json — saved')

# Seedha download kar lo Colab se!
from google.colab import files
files.download('brain.pt')
files.download('vocab.json')
print('\n⬇️  Download shuru ho gaya!')

In [ ]:
# ════════════════════════════════════════════════
# CELL 9 — Loss graph (optional, nice to see)
# ════════════════════════════════════════════════
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(trainer.loss_history, color='steelblue', linewidth=1.5)
plt.title('Training Loss', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=120)
plt.show()
print('✅ loss_curve.png save hua')